In [3]:
import os
from pathlib import Path
import rasterio
import numpy as np
import albumentations as A
from tqdm import tqdm

# Paths
input_img_dir = Path("/home/jovyan/work/training-models/nepal/Nepal_b7_2025/data/imgs_prepared_v2/train_chips")
input_mask_dir = Path("/home/jovyan/work/training-models/nepal/Nepal_b7_2025/data/imgs_prepared_v2/train_multimasks")
output_img_dir = Path("/home/jovyan/work/training-models/nepal/Nepal_b7_2025/data/imgs_prepared_v2/train_chips_aug")
output_mask_dir = Path("/home/jovyan/work/training-models/nepal/Nepal_b7_2025/data/imgs_prepared_v2/train_multimasks_aug")

output_img_dir.mkdir(parents=True, exist_ok=True)
output_mask_dir.mkdir(parents=True, exist_ok=True)

# Number of augmented samples per image
N_AUG = 5

# Photometric augmentation only (georeferencing preserved)
transform = A.Compose([
    # Geometric augmentations
    A.Rotate(limit=30, border_mode=0, interpolation=0, value=0, mask_value=0, p=0.7),  # Rotate like old setup
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    
    # Photometric / color augmentations
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),  # full color jitter
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),  # extra brightness/contrast
    A.GaussianBlur(blur_limit=(3,7), p=0.3),
    A.GaussNoise(var_limit=(10.0,50.0), p=0.3)
], additional_targets={"mask": "mask"})

# Loop over images
img_files = sorted(input_img_dir.glob("*.tif"))

for img_file in tqdm(img_files, desc="Augmenting images"):
    img_name = img_file.stem
    mask_file = input_mask_dir / f"{img_name}.tif"

    # Read image
    try:
        with rasterio.open(img_file) as src:
            img = src.read()  # shape: (bands, H, W)
            profile = src.profile  # save georeference info
            img = np.transpose(img, (1, 2, 0))  # HWC
    except Exception as e:
        print(f"[WARN] Cannot read image {img_file}: {e}")
        continue

    # Read mask
    if mask_file.exists():
        try:
            with rasterio.open(mask_file) as src:
                mask = src.read(1)  # single band
        except Exception as e:
            print(f"[WARN] Cannot read mask {mask_file}: {e}")
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
    else:
        mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

    # Generate augmented samples
    for i in range(N_AUG):
        augmented = transform(image=img, mask=mask)
        aug_img = augmented["image"]
        aug_mask = augmented["mask"]

        # Save image and mask with original georeference
        profile.update(dtype=rasterio.uint8, count=aug_img.shape[2])
        with rasterio.open(output_img_dir / f"{img_name}_aug{i}.tif", "w", **profile) as dst:
            for b in range(aug_img.shape[2]):
                dst.write(aug_img[:,:,b], b+1)

        profile.update(dtype=rasterio.uint8, count=1)
        with rasterio.open(output_mask_dir / f"{img_name}_aug{i}.tif", "w", **profile) as dst:
            dst.write(aug_mask, 1)


Augmenting images:   0%|          | 33/13398 [00:02<14:23, 15.48it/s] 


KeyboardInterrupt: 